# Variant B: rolling one-step forecasting

In variant B each forecast is one step ahead, and the actual value arrives right after. The parameters stay frozen:

1. forecast `t+1` using only what was observed up to `t`;
2. observe `y[t+1]`;
3. update the residual window with the base's error at that moment;
4. move forward, **without calling `fit`**.

Selection uses the one-step error on validation. Dynamic corrector selection is allowed only in this variant.

In [1]:
import autohybts

print("AutoHybTS", autohybts.__version__)

AutoHybTS 0.0.1


## 1. Experiment in variant B

In [2]:
from autohybts import AutoHybridExperiment
from autohybts.datasets import get_data

y = get_data("synthetic_daily")
exp = AutoHybridExperiment(fh=1, variant="B", test_size=28, session_id=42).fit(y)
result = exp.compare_models()
result.leaderboard

,name,model,family,strategy,base,corrector,residual_source,window,alpha,mean_rank,...,MAE,RMSE,sMAPE,coverage,n_units,n_failed,n_skipped,fit_seconds,predict_seconds,is_champion
candidate_id,,,,,,,,,,,,,,,,,,,,,
f197a4a852eefc42b77e9d063dff3c84cf252feda2deb62c878faf84713a4449,arima,arima,statistical,standalone,arima,NaN,NaN,NaN,NaN,1.0,...,0.883120,0.883120,0.012429,1.0,7,0,0,68.098544,0.057547,True
dcc94aa07bedae7630b1f38c2570937852793c439f560bfd9098cbedf9650690,arima-linear,"arima + linear [w=7, alpha=0.5, historical_oof...",residual,residual_additive,arima,linear,historical_oof_1step,7.0,0.5,2.0,...,0.885318,0.885318,0.012459,1.0,7,0,0,25.022806,0.062873,False
0ec158f3b6ec517f088da25b7c2f75c10bfe627d7bf5437f0622b8ff36cca85a,arima-linear,"arima + linear [w=7, alpha=0.5, in_sample]",residual,residual_additive,arima,linear,in_sample,7.0,0.5,3.0,...,0.886370,0.886370,0.012471,1.0,7,0,0,1.669873,0.070572,False
50c3b10b84c20c32b6b898008596592cb4f651f3f73110b54230461381267f3e,arima-linear,"arima + linear [w=7, alpha=1.0, historical_oof...",residual,residual_additive,arima,linear,historical_oof_1step,7.0,1.0,4.0,...,0.891898,0.891898,0.012552,1.0,7,0,0,1.983710,0.064908,False
c51a658f4db43de6316044da15702a7d11e9e319df05363c4688b9ba00547929,mlp,mlp(lags=7),neural,standalone,mlp,NaN,NaN,NaN,NaN,5.0,...,0.892754,0.892754,0.012454,1.0,7,0,0,3.347580,0.009263,False
4a92d125bd10f30ce4cdf0fc07e114c1e0d07bbd03ca06dd5a8c04c2781bfd4b,svm,svm(lags=7),kernel,standalone,svm,NaN,NaN,NaN,NaN,6.0,...,0.926790,0.926790,0.013126,1.0,7,0,0,0.504665,0.013691,False
c05fbfc4191206898692474cb6edb6c95948dcd38fd203e4d6be47b604035efe,linear,linear(lags=7),linear,standalone,linear,NaN,NaN,NaN,NaN,7.0,...,0.939771,0.939771,0.013160,1.0,7,0,0,6.014908,0.012185,False
3e1cf077f43789200b40b04ab5b6098eb7f916132b72ca9bbd6f6f8eba5f3d02,arima-linear,"arima + linear [w=7, alpha=0.5, historical_oof...",residual,residual_additive,arima,linear,historical_oof_multistep,7.0,0.5,8.0,...,0.958163,0.958163,0.013517,1.0,7,0,0,54.950212,0.069409,False
1984f459ddd73973f8dee0312933d0c5d4a4ae9ff5e3e8a400a30bd0b87baff5,linear-linear,"linear(lags=7) + linear [w=7, alpha=0.5, histo...",residual,residual_additive,linear,linear,historical_oof_1step,7.0,0.5,9.0,...,1.002286,1.002286,0.013990,1.0,7,0,0,2.686581,0.010227,False


## 2. Finalize and evaluate on the sealed test

In [3]:
final = exp.finalize_model(result.best)
exp.evaluate_external(final).metrics

,MASE,MAE,RMSE,sMAPE
model,,,,
arima,0.532694,0.83562,1.022446,0.011846


## 3. Streaming in production

The same model, receiving new observations one by one. Here we simulate the arrival of the data with the last 28 observations of the series.

In [4]:
import pandas as pd

stream = final.stream()
rows = []
for timestamp, actual in y.iloc[-28:].items():
    ticket = stream.predict_next()
    rows.append({"timestamp": timestamp, "y_pred": ticket.prediction, "y_true": actual})
    stream.observe(actual)  # only now does the actual value become available

stream_log = pd.DataFrame(rows).set_index("timestamp")
stream_log["abs_error"] = (stream_log["y_true"] - stream_log["y_pred"]).abs()
stream_log

,y_pred,y_true,abs_error
timestamp,,,
2025-12-03,73.163889,72.429605,0.734284
2025-12-04,70.867025,71.246072,0.379047
2025-12-05,68.793807,70.065617,1.271810
2025-12-06,68.582059,67.422345,1.159713
2025-12-07,68.022263,67.361921,0.660342
2025-12-08,69.636127,70.737740,1.101613
2025-12-09,73.005390,73.033376,0.027986
2025-12-10,73.587046,74.455786,0.868740
2025-12-11,72.897800,72.770706,0.127095
